# IBGE Municipalities - Silver Transformation

## Setup

In [0]:
from pyspark.sql.functions import col, trim, lower, translate

environment = "dev"

catalog = f"ecommerce_{environment}"
source_table = f"{catalog}.bronze.ibge_municipalities"
target_table = f"{catalog}.silver.ibge_municipalities"

## Read Bronze data

In [0]:
bronze_df = spark.table(source_table)

## Review the data

In [0]:
bronze_df.printSchema()

root
 |-- ibge_municipality_id: long (nullable = true)
 |-- municipality_name: string (nullable = true)
 |-- state_code: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(bronze_df.limit(10))

ibge_municipality_id,municipality_name,state_code,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
1100015,Alta Floresta D'Oeste,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities
1100023,Ariquemes,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities
1100031,Cabixi,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities
1100049,Cacoal,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities
1100056,Cerejeiras,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities
1100064,Colorado do Oeste,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities
1100072,Corumbiara,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities
1100080,Costa Marques,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities
1100098,Espigão D'Oeste,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities
1100106,Guajará-Mirim,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities


In [0]:
print("Number of rows:", bronze_df.count())
print("Number of columns:", len(bronze_df.columns))

Number of rows: 5571
Number of columns: 9


In [0]:
for column, dtype in bronze_df.dtypes[:3]:
    print(column)
    print("Null count:", bronze_df.filter(col(column).isNull()).count())
    print("Distinct count:", bronze_df.filter(col(column).isNotNull()).select(column).distinct().count())

    if dtype == "string":
        print("Extra whitespace row count:",
            (
                bronze_df.withColumn(f"{column}_trimmed", trim(col(column)))
                .filter(col(column) != col(f"{column}_trimmed"))
                .count()
            )
        )
    print("-"*20)

ibge_municipality_id
Null count: 0
Distinct count: 5571
--------------------
municipality_name
Null count: 0
Distinct count: 5298
Extra whitespace row count: 0
--------------------
state_code
Null count: 0
Distinct count: 27
Extra whitespace row count: 0
--------------------


- ibge_municipality_id is complete and unique. It is the key for this table.

In [0]:
display(bronze_df.select('municipality_name').limit(10))

municipality_name
Alta Floresta D'Oeste
Ariquemes
Cabixi
Cacoal
Cerejeiras
Colorado do Oeste
Corumbiara
Costa Marques
Espigão D'Oeste
Guajará-Mirim


## Transform to Silver

In [0]:
silver_df = bronze_df.withColumn(
    "municipality_name_normalized",
    translate(
        lower(col("municipality_name")),
        "áàâãäéèêëíìîïóòôõöúùûüç-'",
        "aaaaaeeeeiiiiooooouuuuc  "
    )
)

In [0]:
display(silver_df.select('municipality_name_normalized').limit(20))

municipality_name_normalized
alta floresta d oeste
ariquemes
cabixi
cacoal
cerejeiras
colorado do oeste
corumbiara
costa marques
espigao d oeste
guajara mirim


## Write to Silver

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

## Review the result

In [0]:
silver_table_df = spark.table(target_table)

silver_table_df.printSchema()

root
 |-- ibge_municipality_id: long (nullable = true)
 |-- municipality_name: string (nullable = true)
 |-- state_code: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)
 |-- municipality_name_normalized: string (nullable = true)



In [0]:
display(silver_table_df.limit(10))

ibge_municipality_id,municipality_name,state_code,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset,municipality_name_normalized
1100015,Alta Floresta D'Oeste,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities,alta floresta d oeste
1100023,Ariquemes,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities,ariquemes
1100031,Cabixi,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities,cabixi
1100049,Cacoal,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities,cacoal
1100056,Cerejeiras,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities,cerejeiras
1100064,Colorado do Oeste,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities,colorado do oeste
1100072,Corumbiara,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities,corumbiara
1100080,Costa Marques,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities,costa marques
1100098,Espigão D'Oeste,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities,espigao d oeste
1100106,Guajará-Mirim,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities,guajara mirim


In [0]:
print("Bronze row count:", bronze_df.count())
print("Silver row count:", silver_table_df.count())

Bronze row count: 5571
Silver row count: 5571
